# 04 · 端到端 RAG Demo

上传 PDF → VLM 抽表 → 清洗 → 向量库 → 检索 → LLM 回答。

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import config as C
from src.pdf_utils import pdf_to_images, extract_tables
from src.infer import extract_table_from_image, chat
from src.rag import TableVectorStore, parse_markdown_table, build_rag_prompt


## 1. 选一个测试 PDF

In [2]:
pdf = sorted(C.SAMPLES_DIR.glob('*.pdf'))[0]
doc_id = pdf.stem
print(pdf)

/Users/pc-rn/ws/ai-lab/ocr-fine-app/data/samples/apple_2023_q4.pdf


## 2. 两种抽表方式并存：pdfplumber (fast/structured) + VLM (robust/image)

In [3]:
# 方式 A：pdfplumber (快，有结构时最准)
pp_tables = extract_tables(pdf)
print(f'pdfplumber 抽到 {len(pp_tables)} 张表')
if pp_tables:
    print(pp_tables[0]['markdown'][:300])

pdfplumber 抽到 3 张表
| Products | ! 67,184 |  | ! 70,958 |  | ! 298,085 |  | ! 316,199 |
|---|---|---|---|---|---|---|---|
| Services | 22,314 |  | 19,188 |  | 85,200 |  | 78,129 |
| Total net sales (1) | 89,498 |  | 90,146 |  | 383,285 |  | 394,328 |
| Cost of sales: |  |  |  |  |  |  |  |
| Products | 42,586 |  | 46,3


In [4]:
# 方式 B：VLM 从图片抽（扫描件或结构异常时更鲁棒）
imgs = pdf_to_images(pdf, C.DATA_DIR / 'preview' / doc_id)
vlm_md = extract_table_from_image(imgs[0], adapter=str(C.STAGE1_ADAPTER), max_tokens=512)
print(vlm_md[:500])

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

FileNotFoundError: The adapter path does not exist: /Users/pc-rn/ws/ai-lab/ocr-fine-app/models/stage1_adapter

## 3. 写入向量库

In [ ]:
store = TableVectorStore()
for t in pp_tables:
    df = parse_markdown_table(t['markdown'])
    if not df.empty:
        n = store.add(df, doc_id=doc_id, page=t['page'])
        print(f'p{t["page"]}  +{n} chunks')
print('总 chunks:', store.count())

## 4. RAG 问答

In [ ]:
question = '营收最高的是哪一年，具体是多少？'
hits = store.search(question, top_k=5, doc_filter=doc_id)
for h in hits: print(f'{h["score"]:.2f}  {h["text"][:120]}')

msgs = build_rag_prompt(question, hits)
answer = chat(msgs, adapter=str(C.STAGE2_ADAPTER), max_tokens=400)
print('\n=== 回答 ===')
print(answer)

## 5. 打开 Streamlit UI

```bash
streamlit run app/streamlit_app.py
```